In [ ]:
import pandas as pd
import os
import glob

In [ ]:
def extract_time_info(ts_int):
    total_hm = ts_int // 100
    
    hour = total_hm // 100
    minute = total_hm % 100
    
    total_minutes = hour * 60 + minute
    
    return hour, minute, total_minutes

In [ ]:
def timetable_2_pieceofwork(file_path):

    timetable_raw = pd.read_csv(file_path)

    timetable_raw['timestamp'] = pd.to_numeric(timetable_raw['timestamp'], errors='coerce')
    
    timetable_piece = timetable_raw.sort_values(['trip_index', 'timestamp']).groupby('trip_index').agg(
        start_stop=('stop_name', 'first'),
        end_stop=('stop_name', 'last'),
        start_time=('timestamp', 'first'),
        end_time=('timestamp', 'last')
    ).reset_index()

    timetable_piece[['start_hour', 'start_min', 'start_total_min']] = timetable_piece['start_time'].apply(
        lambda x: pd.Series(extract_time_info(x))
    )

    timetable_piece[['end_hour', 'end_min', 'end_total_min']] = timetable_piece['end_time'].apply(
        lambda x: pd.Series(extract_time_info(x))
    )

    timetable_piece['duration_min'] = timetable_piece['end_total_min'] - timetable_piece['start_total_min']

    timetable_output = timetable_piece.drop(columns=["start_stop","end_stop","start_time","end_time","start_total_min","end_total_min"])

    return timetable_output

In [ ]:
def folder_runthrought(input_dir, output_dir):
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    file_pattern = os.path.join(input_dir, "uov_timetable_*.csv")
    input_files = glob.glob(file_pattern)
    
    for file_path in input_files:

        base_name = os.path.basename(file_path)
        
        output_name = base_name.replace("timetable", "pow")
        output_path = os.path.join(output_dir, output_name)
        
        print(f"processing: {base_name} -> {output_name}")
        
        # read file and call timetable_2_pieceofwork
        try:
            print(file_path)
            
            df_pow = timetable_2_pieceofwork(file_path)
            
            df_pow.to_csv(output_path, index=False)
            
        except Exception as e:
            print(f"An error occurred while processing {base_name}: {e}")

In [ ]:
if __name__ == "__main__":
    input_folder = 'timetable_bus1to8'
    output_folder = 'pow_bus1to8'
    folder_runthrought(input_folder, output_folder)